# Kaggle: cobertura para Itinerante
Perfil reproducible de dos fuentes descargadas el 2026-09-04.
Ejecutar desde este repositorio. Conserva los originales en imports; no escribe en la base de datos.
Los géneros son etiquetas de origen y los conteos por tema se superponen.


In [ ]:
from pathlib import Path
import os
project = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "imports/kaggle-goodreads").exists())
os.chdir(project)


In [1]:
"""Read-only source profile. Run from the Itinerante project root."""
import csv
import hashlib
import io
import json
import re
import unicodedata
import zipfile
from collections import Counter
from pathlib import Path

BASE = Path('imports/kaggle-books-es')
EXPECTED_BYTES = 23127876  # Kaggle API file listing, retrieved 2026-09-04


def normalize(value):
    return ' '.join(''.join(c for c in unicodedata.normalize('NFD', str(value or ''))
                           if unicodedata.category(c) != 'Mn').lower().split())


def isbn_valid(value):
    value = re.sub(r'[\s-]', '', str(value or '')).upper()
    if len(value) == 13 and value.isdigit():
        return sum(int(c) * (1 if i % 2 == 0 else 3) for i, c in enumerate(value)) % 10 == 0
    if len(value) == 10 and value[:9].isdigit() and (value[-1].isdigit() or value[-1] == 'X'):
        return sum((10 - i) * (10 if c == 'X' else int(c)) for i, c in enumerate(value)) % 11 == 0
    return False


def profile():
    path = BASE / 'books.download'
    if zipfile.is_zipfile(path):
        with zipfile.ZipFile(path) as archive:
            name = next(n for n in archive.namelist() if n.endswith('books.csv'))
            raw = archive.read(name)
    else:
        raw = path.read_bytes()
    assert len(raw) == EXPECTED_BYTES, f'Incomplete or changed source: {len(raw)} bytes'
    reader = csv.DictReader(io.StringIO(raw.decode('utf-8-sig')))
    rows = list(reader)
    assert rows and {'title', 'writer', 'gender', 'isbn', 'description'} <= set(reader.fieldnames)
    missing = lambda value: normalize(value) in {'', 'nan', 'null', 'none'}
    author_seeds = ['Juan Rulfo', 'Octavio Paz', 'Rosario Castellanos', 'Elena Poniatowska',
                    'Carlos Fuentes', 'Sor Juana', 'Amado Nervo', 'Jaime Sabines',
                    'José Emilio Pacheco', 'Elena Garro', 'Alfonso Reyes',
                    'Valeria Luiselli', 'Fernanda Melchor']
    genres = Counter(r['gender'] for r in rows)
    topics = {
        label: [r for r in rows if pattern in normalize(r['gender'])]
        for label, pattern in {'poesia': 'poe', 'psicologia': 'psicolog', 'filosofia': 'filosof'}.items()
    }
    authors = {name: [r for r in rows if normalize(name) in normalize(r['writer'])]
               for name in author_seeds}
    pairs = Counter((normalize(r['title']), normalize(r['writer'])) for r in rows)
    isbns = Counter(re.sub(r'[\s-]', '', r['isbn']) for r in rows if not missing(r['isbn']))
    descriptions = Counter(normalize(r['description']) for r in rows if not missing(r['description']))
    sample = lambda records: [dict(title=r['title'], writer=r['writer'], isbn=r['isbn'],
                                   gender=r['gender']) for r in records[:4]]
    result = {
        'source_url': 'https://www.kaggle.com/datasets/danielalbarracinm/books-with-isbn-of-different-genres',
        'retrieved_on': '2026-09-04', 'csv_bytes': len(raw),
        'csv_sha256': hashlib.sha256(raw).hexdigest(),
        'rows': len(rows), 'columns': reader.fieldnames,
        'missing_by_column': {k: sum(missing(r[k]) for r in rows) for k in reader.fieldnames},
        'exact_duplicate_excess': len(rows) - len({tuple(r.items()) for r in rows}),
        'title_author_duplicate_excess': sum(n - 1 for n in pairs.values()),
        'isbn_duplicate_excess': sum(n - 1 for n in isbns.values()),
        'valid_isbn_rows': sum(isbn_valid(r['isbn']) for r in rows),
        'invalid_isbn_examples': [r['isbn'] for r in rows if not isbn_valid(r['isbn'])][:10],
        'repeated_description_excess': sum(n - 1 for n in descriptions.values()),
        'description_under_80_chars': sum(len(r['description'].strip()) < 80 for r in rows),
        'genres': dict(genres.most_common()),
        'topic_counts_by_source_genre': {k: len(v) for k, v in topics.items()},
        'topic_examples': {k: sample(v) for k, v in topics.items()},
        'seed_author_counts': {k: len(v) for k, v in authors.items()},
        'seed_author_examples': {k: sample(v) for k, v in authors.items() if v},
        'limitations': [
            'Author seeds are a coverage probe, not a complete nationality classification.',
            'Title-author duplicates may be distinct editions; never delete solely on this key.',
            'A valid ISBN checksum does not verify correspondence with the book.',
            'Genre and description values have not received full editorial verification.',
            'Bibliographic records provide no evidence of Itinerante stock or price.'
        ]
    }
    (BASE / 'profile.json').write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding='utf-8')
    return result


if __name__ == '__main__':
    assert isbn_valid('9780306406157')
    assert not isbn_valid('9780306406158')
    result = profile()
    print(json.dumps(result, ensure_ascii=False, indent=2))


{
  "source_url": "https://www.kaggle.com/datasets/danielalbarracinm/books-with-isbn-of-different-genres",
  "retrieved_on": "2026-09-04",
  "csv_bytes": 23127876,
  "csv_sha256": "b1566fd93b69ebaef29a1dcaab1173384d5fd36a939c353f22d44611d4e7b6b6",
  "rows": 31284,
  "columns": [
    "title",
    "description",
    "gender",
    "num_pages",
    "editorial",
    "isbn",
    "year_edition",
    "date_edition",
    "writer",
    "image"
  ],
  "missing_by_column": {
    "title": 4,
    "description": 8087,
    "gender": 58,
    "num_pages": 0,
    "editorial": 15977,
    "isbn": 38,
    "year_edition": 0,
    "date_edition": 18161,
    "writer": 22134,
    "image": 38
  },
  "exact_duplicate_excess": 5636,
  "title_author_duplicate_excess": 7312,
  "isbn_duplicate_excess": 5645,
  "valid_isbn_rows": 31011,
  "invalid_isbn_examples": [
    "",
    "",
    "",
    "",
    "",
    "",
    "",
    "",
    "",
    ""
  ],
  "repeated_description_excess": 5153,
  "description_under_80_chars": 9

In [2]:
"""Profile the Kaggle source and stage a bibliographic subset; never imports inventory."""
import ast
import csv
import hashlib
import importlib.util
import io
import json
import zipfile
from collections import Counter
from pathlib import Path

BASE = Path('imports/kaggle-goodreads')
SOURCE = 'https://www.kaggle.com/datasets/thedevastator/comprehensive-overview-of-52478-goodreads-best-b'
helpers_spec = importlib.util.spec_from_file_location('source_quality', 'imports/kaggle-books-es/profile.py')
helpers = importlib.util.module_from_spec(helpers_spec)
helpers_spec.loader.exec_module(helpers)
normalize, isbn_valid = helpers.normalize, helpers.isbn_valid


def profile():
    with zipfile.ZipFile(BASE / 'books.download') as archive:
        raw = archive.read('books_1.Best_Books_Ever.csv')
    assert len(raw) == 73839808, 'Source changed or download incomplete'
    reader = csv.DictReader(io.StringIO(raw.decode('utf-8-sig')))
    rows = list(reader)
    seeds = ['Juan Rulfo', 'Octavio Paz', 'Rosario Castellanos', 'Elena Poniatowska',
             'Carlos Fuentes', 'Sor Juana', 'Amado Nervo', 'Jaime Sabines',
             'José Emilio Pacheco', 'Elena Garro', 'Alfonso Reyes',
             'Valeria Luiselli', 'Fernanda Melchor']
    topics = ['Poetry', 'Psychology', 'Philosophy', 'Mexican Literature']
    for row in rows:
        row['_genres'] = ast.literal_eval(row['genres']) if row['genres'] else []
        assert isinstance(row['_genres'], list) and all(isinstance(g, str) for g in row['_genres'])
        row['_author'] = normalize(row['author'])
    missing = lambda value: normalize(value) in {'', 'nan', 'null', 'none'}
    spanish = [r for r in rows if normalize(r['language']) == 'spanish']
    coverage = {name: [r for r in rows if r['_author'].startswith(normalize(name))] for name in seeds}
    sample = lambda records: [{k: r[k] for k in ['bookId', 'title', 'author', 'language', 'isbn', 'genres']}
                              for r in records[:4]]
    subset = [r for r in rows if
              (normalize(r['language']) == 'spanish' and any(t in r['_genres'] for t in topics))
              or 'Mexican Literature' in r['_genres']
              or any(r['_author'].startswith(normalize(name)) for name in seeds)]
    # Preserve each source row; duplicate resolution belongs to a reviewed import.
    fields = ['bookId', 'title', 'author', 'description', 'language', 'isbn', 'genres',
              'bookFormat', 'edition', 'pages', 'publisher', 'publishDate', 'coverImg']
    staged = [{k: r[k] for k in fields} for r in subset]
    (BASE / 'bibliographic-candidates.json').write_text(json.dumps({
        'source': SOURCE, 'sourceCsvSha256': hashlib.sha256(raw).hexdigest(),
        'purpose': 'Bibliographic discovery candidates; not store inventory; not yet imported.',
        'selection': 'Spanish Poetry/Psychology/Philosophy, Mexican Literature shelf, or one of 13 named first-contributor probes.',
        'books': staged
    }, ensure_ascii=False, indent=2), encoding='utf-8')
    isbns = Counter(r['isbn'] for r in rows if not missing(r['isbn']))
    ids = Counter(r['bookId'] for r in rows)
    pairs = Counter((normalize(r['title']), r['_author']) for r in rows)
    result = {
        'source_url': SOURCE, 'retrieved_on': '2026-09-04', 'csv_bytes': len(raw),
        'csv_sha256': hashlib.sha256(raw).hexdigest(), 'rows': len(rows), 'columns': reader.fieldnames,
        'languages': dict(Counter(r['language'] for r in rows).most_common()),
        'missing_by_column': {k: sum(missing(r[k]) for r in rows) for k in reader.fieldnames},
        'source_id_duplicate_excess': sum(n - 1 for n in ids.values()),
        'isbn_duplicate_excess': sum(n - 1 for n in isbns.values()),
        'title_author_duplicate_excess': sum(n - 1 for n in pairs.values()),
        'valid_isbn_rows': sum(isbn_valid(r['isbn']) for r in rows),
        'isbn_sentinel_9999999999999': sum(r['isbn'] == '9999999999999' for r in rows),
        'empty_genre_lists': sum(not r['_genres'] for r in rows),
        'topic_counts': {t: sum(t in r['_genres'] for r in rows) for t in topics},
        'spanish_topic_counts': {t: sum(t in r['_genres'] for r in spanish) for t in topics},
        'spanish_topic_examples': {t: sample([r for r in spanish if t in r['_genres']]) for t in topics},
        'seed_author_counts': {name: len(records) for name, records in coverage.items()},
        'seed_author_examples': {name: sample(records) for name, records in coverage.items() if records},
        'subset_rows': len(subset),
        'subset_missing_description': sum(missing(r['description']) for r in subset),
        'subset_languages': dict(Counter(r['language'] for r in subset).most_common()),
        'subset_source_id_duplicate_excess': len(subset) - len({r['bookId'] for r in subset}),
        'limitations': [
            'Source genres are community tags, not an authoritative subject classification.',
            'Counts overlap: a book can have several genres.',
            'Seed-author counts match the first contributor only; this is a partial probe, not a nationality census.',
            'Language is the source field, not an independent language detection result.',
            'The staged subset preserves source duplicates for review and has no store price or availability.'
        ]
    }
    (BASE / 'profile.json').write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding='utf-8')
    return result


if __name__ == '__main__':
    result = profile()
    print(json.dumps(result, ensure_ascii=False, indent=2))


{
  "source_url": "https://www.kaggle.com/datasets/thedevastator/comprehensive-overview-of-52478-goodreads-best-b",
  "retrieved_on": "2026-09-04",
  "csv_bytes": 73839808,
  "csv_sha256": "6b56f444a9c26fa643b41fb6b6a9248af3d81f3812e11e25595d5a65b6340453",
  "rows": 52478,
  "columns": [
    "bookId",
    "title",
    "series",
    "author",
    "rating",
    "description",
    "language",
    "isbn",
    "genres",
    "characters",
    "bookFormat",
    "edition",
    "pages",
    "publisher",
    "publishDate",
    "firstPublishDate",
    "awards",
    "numRatings",
    "ratingsByStars",
    "likedPercent",
    "setting",
    "coverImg",
    "bbeScore",
    "bbeVotes",
    "price"
  ],
  "languages": {
    "English": 42661,
    "": 3806,
    "Arabic": 1038,
    "Spanish": 687,
    "French": 579,
    "German": 528,
    "Indonesian": 371,
    "Portuguese": 363,
    "Italian": 255,
    "Dutch": 215,
    "Turkish": 211,
    "Persian": 164,
    "Polish": 138,
    "Greek, Modern (1453-)": 